# Meaningful Multivariate Subsequence Search

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/stumpy-dev/stumpy/main?filepath=docs/Tutorial_Multivariate_Subsequence_Search.ipynb)

## Finding Top-k Matches Across Channels With Different Units

This tutorial is inspired by Eamonn Keogh's mini tutorial, [Meaningful Fast Top-k Subsequence Search for Multivariate Time Series](https://www.dropbox.com/scl/fi/vs0vei3i76wfd0m32n0cl/Meaningful-Fast-Top-k-Subsequence-Search-for-Multivariate-Time.pdf?rlkey=mm57cn45101f7pueytbq66dr6&e=1&dl=0), and the accompanying [source post](https://www.linkedin.com/posts/eamonn-keogh-96ab25143_timeseriesanalysis-patternmining-machinelearning-activity-7413378995796439040-b_ET). The central point is practical: raw subsequence distances can become meaningless when multivariate channels use different units or scales.

Here, we will use `stumpy.match` to search for the top-k multivariate subsequence matches. We will first show how raw Euclidean distance can be distracted by a large-scale but irrelevant channel. Then we will use STUMPY's default z-normalized search and select only the query-relevant channels.

## Getting Started

Let's import the packages that we'll need to generate, analyze, and plot a small synthetic dataset.

In [ ]:
%matplotlib inline

import numpy as np
import stumpy
import matplotlib.pyplot as plt

for style in ("./stumpy.mplstyle", "docs/stumpy.mplstyle"):
    try:
        plt.style.use(style)
        break
    except OSError:
        pass

## Creating Flight-like Sensor Data

Suppose that we are searching through a multivariate time series collected from a flight. The channels have different physical units: altitude, airspeed, outside temperature, and hydraulic pressure. Three short maneuver windows share the same shape in the first three channels. The pressure channel, however, contains a large pressure pulse in the query window and in one unrelated window.

This is intentionally constructed so that raw distance can prefer the pressure-only distractor over the true repeated maneuvers.

In [ ]:
def make_flight_sensor_data(seed=12):
    rng = np.random.default_rng(seed)
    n = 720
    m = 64
    x = np.arange(n)

    T = np.vstack(
        [
            32000 + 50 * np.sin(x / 70) + rng.normal(0, 4, n),
            450 + 3 * np.sin(x / 35) + rng.normal(0, 0.5, n),
            -40 + 1.0 * np.sin(x / 50) + rng.normal(0, 0.15, n),
            3000 + 120 * np.sin(x / 45) + rng.normal(0, 18, n),
        ]
    )

    u = np.linspace(0, 1, m)
    maneuver = np.vstack(
        [
            -120 * np.sin(np.pi * u),
            -8 * np.sin(np.pi * u) + 2 * np.sin(2 * np.pi * u),
            -2.0 * np.sin(np.pi * u),
        ]
    )

    maneuver_starts = np.array([112, 336, 552])
    for scale, start in zip([0.97, 1.0, 1.05], maneuver_starts):
        T[:3, start : start + m] += scale * maneuver

    pressure_shape = 1800 * np.sin(np.pi * u) + 400 * np.sin(2 * np.pi * u)
    pressure_starts = np.array([336, 232])
    for start in pressure_starts:
        T[3, start : start + m] += pressure_shape

    channel_names = [
        "altitude (ft)",
        "airspeed (knots)",
        "temperature (deg C)",
        "hydraulic pressure (psi)",
    ]
    known_events = {
        "similar maneuver 1": int(maneuver_starts[0]),
        "query maneuver": int(maneuver_starts[1]),
        "similar maneuver 2": int(maneuver_starts[2]),
        "pressure-only distractor": int(pressure_starts[1]),
    }
    relevant_dims = np.array([0, 1, 2])

    return T, m, int(maneuver_starts[1]), relevant_dims, channel_names, known_events

In [ ]:
T, m, query_idx, relevant_dims, channel_names, known_events = make_flight_sensor_data()
Q = T[:, query_idx : query_idx + m]

T.shape, Q.shape, m, query_idx

Each row in `T` is a sensor channel, and each column is a time point. The query `Q` is the window that starts at `query_idx`.

In [ ]:
def shade_known_windows(axs, known_events, m):
    colors = {
        "similar maneuver 1": "C2",
        "query maneuver": "C1",
        "similar maneuver 2": "C2",
        "pressure-only distractor": "C3",
    }

    for label, start in known_events.items():
        for ax in axs:
            ax.axvspan(start, start + m, color=colors[label], alpha=0.12)

    for label, start in known_events.items():
        axs[0].text(
            start,
            1.03,
            label,
            rotation=90,
            va="bottom",
            fontsize=9,
            transform=axs[0].get_xaxis_transform(),
        )


fig, axs = plt.subplots(
    T.shape[0], 1, figsize=(14, 8), sharex=True, gridspec_kw={"hspace": 0.15}
)

for ax, channel, name in zip(axs, T, channel_names):
    ax.plot(channel, lw=1)
    ax.set_ylabel(name)

shade_known_windows(axs, known_events, m)
axs[-1].set_xlabel("Time")
axs[0].set_title("Synthetic Multivariate Sensor Data")
plt.show()

## The Query

The query is the highlighted maneuver at time `336`. We will ask STUMPY to find the closest matching windows in the full time series.

In [ ]:
fig, axs = plt.subplots(
    Q.shape[0], 1, figsize=(10, 7), sharex=True, gridspec_kw={"hspace": 0.15}
)

for ax, channel, name in zip(axs, Q, channel_names):
    ax.plot(channel, lw=2, color="C1")
    ax.set_ylabel(name)

axs[-1].set_xlabel("Relative time")
axs[0].set_title("Multivariate Query Window")
plt.show()

## A Small Reporting Helper

When the query is extracted from the same time series that we are searching, the closest match is the query itself. So, if we want the top `k` neighbors in addition to the self-match, we ask for `k + 1` matches and pass `query_idx`.

In [ ]:
def nearest_event(idx, known_events, tolerance=8):
    distance, label = min(
        (abs(idx - start), label) for label, start in known_events.items()
    )
    if distance <= tolerance:
        return label
    return ""


def print_matches(title, matches, known_events, limit=None):
    print(title)
    print("rank  start  distance  nearest known event")
    for rank, (distance, idx) in enumerate(matches[:limit], start=1):
        idx = int(idx)
        print(
            f"{rank:>4}  {idx:>5}  {float(distance):>8.3f}  "
            f"{nearest_event(idx, known_events)}"
        )

## Raw Multivariate Search Can Be Misleading

First, let's turn z-normalization off by setting `normalize=False`. This computes a raw, non-normalized distance across the channels. Since hydraulic pressure is measured on a much larger numeric scale than the other channels, it can dominate the distance.

In [ ]:
raw_matches = stumpy.match(
    Q,
    T,
    max_matches=6,
    max_distance=np.inf,
    query_idx=query_idx,
    normalize=False,
)

print_matches("Raw non-normalized matches", raw_matches, known_events)

The first row is the self-match. The next row is the pressure-only distractor, not one of the other two maneuver windows. This is the failure mode: raw distance found a numerically similar pressure pulse, even though the maneuver channels do not match well.

In [ ]:
def plot_query_and_matches(Q, T, matches, dims, channel_names, title, max_neighbors=3):
    neighbors = [(float(distance), int(idx)) for distance, idx in matches][
        1 : 1 + max_neighbors
    ]
    fig, axs = plt.subplots(
        len(dims),
        1,
        figsize=(12, 2.6 * len(dims)),
        sharex=True,
        gridspec_kw={"hspace": 0.15},
    )
    axs = np.atleast_1d(axs)
    x = np.arange(Q.shape[-1])

    for ax, dim in zip(axs, dims):
        ax.plot(x, Q[dim], color="black", lw=2, label="query")
        for rank, (_, idx) in enumerate(neighbors, start=2):
            ax.plot(
                x,
                T[dim, idx : idx + Q.shape[-1]],
                lw=1.5,
                alpha=0.8,
                label=f"rank {rank}: start {idx}",
            )
        ax.set_ylabel(channel_names[dim])

    axs[0].set_title(title)
    axs[-1].set_xlabel("Relative time")
    axs[0].legend(loc="upper right")
    plt.show()


plot_query_and_matches(
    Q,
    T,
    raw_matches,
    np.arange(T.shape[0]),
    channel_names,
    "Raw Search: Top Matches After the Self-match",
)

## Z-normalized Multivariate Search With `stumpy.match`

By default, `stumpy.match` z-normalizes subsequences before computing distances. This compares shape rather than raw units, which is usually what we want for subsequence search across channels with different scales.

In [ ]:
znormalized_matches = stumpy.match(
    Q,
    T,
    max_matches=6,
    max_distance=np.inf,
    query_idx=query_idx,
)

print_matches("Z-normalized matches across all channels", znormalized_matches, known_events)

The true maneuver windows are now retrieved before the pressure-only distractor. The pressure channel no longer dominates just because its numbers are larger.

In [ ]:
plot_query_and_matches(
    Q,
    T,
    znormalized_matches,
    np.arange(T.shape[0]),
    channel_names,
    "Z-normalized Search Across All Channels",
)

## Query-time Channel Selection

In many multivariate searches, not every channel is relevant to the question. If the analyst knows that the query is about the maneuver shape, then altitude, airspeed, and temperature are relevant while the pressure pulse is not. We can express that directly by passing only those rows to `stumpy.match`.

In [ ]:
selected_matches = stumpy.match(
    Q[relevant_dims],
    T[relevant_dims],
    max_matches=6,
    max_distance=np.inf,
    query_idx=query_idx,
)

print_matches("Z-normalized matches on selected channels", selected_matches, known_events)

This is a query-time choice. We did not build an index or preprocess a special version of the data. We simply selected the rows that matter for the current search.

In [ ]:
plot_query_and_matches(
    Q,
    T,
    selected_matches,
    relevant_dims,
    channel_names,
    "Z-normalized Search on Selected Channels",
)

## Extracting the Top-k Neighbors

The `max_matches` argument controls how many matches are returned. Since our query is a subsequence of `T`, the first row is the self-match. The top `k` non-trivial neighbors are the next `k` rows.

In [ ]:
k = 2
top_k_neighbors = selected_matches[1 : k + 1]

print_matches(f"Top {k} non-trivial neighbors", top_k_neighbors, known_events)

## Summary

- Raw multivariate subsequence distance can be dominated by whichever channel has the largest numeric scale.
- `stumpy.match` uses z-normalized subsequence distances by default, which makes shape-based search meaningful across channels with different units.
- For query-time channel selection, pass only the relevant rows of `Q` and `T`.
- Use `max_matches` for top-k search, and request one extra match when the query is itself a subsequence of the searched time series.